In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

In [ ]:
# Tampilkan data secara penuh tanpa terpotong
spark.sql("SELECT * FROM raw_table LIMIT 5").show(truncate=False)

In [24]:
# 1. Inisialisasi IP
IP_KAFKA_NAD = "100.75.210.119"       
IP_CASSANDRA_IAN = "100.66.223.98"  

# 2. Inisialisasi Spark
spark = SparkSession.builder \
    .appName("Dedolarisasi-StreamReader") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,com.datastax.spark:spark-cassandra-connector_2.12:3.5.0") \
    .config("spark.cassandra.connection.host", IP_CASSANDRA_IAN) \
    .getOrCreate()

In [25]:
# 3. Definisikan Skema Data Mentah (JSON dari Kafka)
schema = StructType([
    StructField("currency_pair", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", LongType(), True)
])
# 4. Baca Aliran Data dari Kafka
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", f"{IP_KAFKA_NAD}:29092") \
    .option("subscribe", "forex-raw") \
    .option("startingOffsets", "latest") \
    .load()

In [26]:
# 5. Parse JSON dan sesuaikan tipe kolom timestamp ke Cassandra
parsed_df = kafka_df \
    .selectExpr("CAST(value AS STRING) as json_value") \
    .select(from_json(col("json_value"), schema).alias("data")) \
    .select("data.*") \
    .withColumn("ts", col("timestamp").cast("timestamp")) \
    .drop("timestamp")
# 6. Fungsi untuk menulis data streaming ke Cassandra per micro-batch
def write_to_cassandra(batch_df, batch_id):
    batch_df.write \
        .format("org.apache.spark.sql.cassandra") \
        .options(table="forex_rates", keyspace="dedolarisasi") \
        .mode("append") \
        .save()

In [27]:
# 7. Jalankan Stream
query = parsed_df.writeStream \
    .foreachBatch(write_to_cassandra) \
    .start()
query.awaitTermination()

StreamingQueryException: [STREAM_FAILED] Query [id = 3f951808-1766-4b4b-bb1b-00752cdd4218, runId = 4c9376cf-ee17-43ff-9720-dc3ca83fe16b] terminated with exception: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/pyspark/sql/utils.py", line 120, in call
    raise e
  File "/usr/local/spark/python/pyspark/sql/utils.py", line 117, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/tmp/ipykernel_3047/1094853098.py", line 14, in write_to_cassandra
    .save()
     ^^^^^^
  File "/usr/local/spark/python/pyspark/sql/readwriter.py", line 1461, in save
    self._jwrite.save()
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1

In [31]:
# Jalankan ini untuk mengintip data asli dari Kafka Jimly
raw_query = kafka_df.selectExpr("CAST(value AS STRING) as raw_json") \
    .writeStream \
    .format("console") \
    .start()